# Madrid Salesas demo: download and analysis on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JSempereH/360_city_analyser/blob/main/notebooks/colab_madrid_salesas.ipynb)

Prepares the data for a second presentation demo with sharper imagery than Valencia: the Salesas and Chueca
neighborhood of Madrid, captured in October 2026 with a GoPro MAX2 at 7680x3840. The notebook:

1. downloads about 160 of those panoramas at their original resolution, 10 m apart, leaving out older captures;
2. analyzes them on the GPU with the settings of the other demos (`balanced`, tiles of 384 px);
3. packs the results, the panoramas and the model-stage cache into two zip files on Drive.

The answers per building, the mask review and the curation are done afterwards on the laptop, as for Valencia.
Nothing here reads reference data: the cadastre is only matched later, for evaluation.

**Before running it**

- Runtime > Change runtime type > **T4 GPU** (or better).
- A Colab secret `MAPILLARY_ACCESS_TOKEN` with notebook access: the client token, `MLY|<app id>|<32 hex>`.
- The repository on GitHub must include `--captured-after` in `city-analyser-download` (commit and push first).

Run the cells in order. Every command stops the notebook with its own error when it fails, so the first red cell is
the one to read. Running a cell again resumes: downloaded files and finished analyses are skipped.

## 1. Drive, repository and environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/city-analyser'      # datasets, caches and the results persist here
os.makedirs(WORK + '/data', exist_ok=True)

In [ ]:
import subprocess

def sh(command, cwd='/content/repo'):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print('$', command, flush=True)
    process = subprocess.Popen(command, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    if process.wait() != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}: {command}')

def secret(name):
    """A Colab secret (key icon on the left; enable notebook access), or None."""
    try:
        from google.colab import userdata
        return (userdata.get(name) or '').strip().strip('"').strip("'") or None  # pasted values often carry spaces or quotes
    except Exception:
        return None

In [ ]:
# A public repository needs no token. For a private fork, store a read-only token in Colab secrets as GITHUB_TOKEN.
REPOSITORY = 'github.com/JSempereH/360_city_analyser.git'
token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPOSITORY}' if token else f'https://{REPOSITORY}'
sh('rm -rf /content/repo', cwd='/content')
sh(f'git clone --depth 1 {url} /content/repo', cwd='/content')
sh("grep -q -- '--captured-after' src/city_analyser/mapillary_street_download.py || "
   "(echo 'This checkout has no --captured-after: commit and push the downloader change first.' && exit 1)")

In [ ]:
sh('pip -q install uv')
sh('uv sync --extra cu126')
# data/ lives on Drive. Model weights stay on the session disk: Drive does not keep the Hugging Face cache's
# symlinks and can serve a file still being synced, which breaks loading (SafetensorError: header too large).
sh(f'rm -rf data models && ln -s {WORK}/data data && mkdir -p /content/models && ln -s /content/models models')
sh('uv run python -c "import torch; assert torch.cuda.is_available(), \'no GPU: Runtime > Change runtime type\'; print(torch.cuda.get_device_name(0))"')

In [ ]:
DATASET = 'madrid-salesas'
# The settings of the presentation demos (Makefile PRESENTATION_ENV): the laptop must use the same ones,
# or it treats these analyses as stale.
os.environ['BUILDING_ANALYSIS_MODEL'] = 'balanced'
os.environ['BUILDING_ANALYSIS_TILE_SIZE'] = '384'
# The model outputs, kept apart so they can be taken to the laptop: a later change to the association
# (a new ANALYSIS_VERSION) then re-associates there in seconds instead of running the models on a CPU.
os.environ['BUILDING_ANALYSIS_STAGE_CACHE_DIR'] = f'{WORK}/stage-cache-{DATASET}'
os.environ['BUILDING_ANALYSIS_OSM_CACHE_DIR'] = WORK + '/osm-cache'
sh('uv run city-analyser-models')

In [ ]:
# The laptop keys its stage cache by these revisions. If one differs, run `uv run city-analyser-models`
# on the laptop before importing, so it has the same weights and reuses the cache.
LAPTOP = {
    'nvidia/segformer-b5-finetuned-cityscapes-1024-1024': '2c6f153e4c23c229e2fa2b188eb250607e030cd8',
    'IDEA-Research/grounding-dino-tiny': 'a2bb814dd30d776dcf7e30523b00659f4f141c71',
    'facebook/sam2.1-hiera-small': 'ee5bba1d82bb8749febdf90f45e84b687142ba03',
}
open('/content/revisions.py', 'w').write("""
from pathlib import Path
from city_analyser import building_analysis as ba
from city_analyser.building_instances import instance_settings
model, _ = ba._segmentation_model_spec()
settings = instance_settings()
for name in (model, settings.detector, settings.segmenter):
    print(name, ba._cached_model_revision(name, Path('models'), None))
""")
sh('uv run python /content/revisions.py > /content/revisions.txt')
for line in open('/content/revisions.txt'):
    name, revision = line.split()
    expected = LAPTOP.get(name)
    note = 'same as the laptop' if revision == expected else f'DIFFERENT from the laptop ({expected}): run city-analyser-models there'
    print(f'{name}: {revision[:12]}  {note}')

## 2. Download

The area is a rectangle around Salesas and Chueca (about 600 x 600 m). `--captured-after 2026-10-01` keeps only the
GoPro MAX2 sequences: without it an older, softer capture at the same spot would be chosen first.
`--min-spacing-m 10` gives about 160 panoramas, some 1.5 to 2 GB at full resolution.

In [ ]:
BBOX = '-3.6960,40.4250,-3.6890,40.4305'   # min_lon,min_lat,max_lon,max_lat
mapillary_token = secret('MAPILLARY_ACCESS_TOKEN')
if not mapillary_token:
    raise RuntimeError('No MAPILLARY_ACCESS_TOKEN secret: add it under the key icon and enable notebook access.')
import urllib.error, urllib.parse, urllib.request
try:  # one tiny query first, so a wrong token fails here with a clear message
    urllib.request.urlopen('https://graph.mapillary.com/images?' + urllib.parse.urlencode({'access_token': mapillary_token, 'bbox': BBOX, 'fields': 'id', 'limit': 1}), timeout=30)
except urllib.error.HTTPError as error:
    if error.code in (400, 401, 403):
        raise RuntimeError(f'Mapillary rejected the MAPILLARY_ACCESS_TOKEN secret (HTTP {error.code}). It must be the client access token, '
                           'MLY|<app id>|<32 hex characters>, from mapillary.com/dashboard/developers, not the client secret.') from None
    raise
os.environ['MAPILLARY_ACCESS_TOKEN'] = mapillary_token
sh(f'uv run city-analyser-download --tiled-bbox={BBOX} --output data/{DATASET} --max-images 400 --min-spacing-m 10 '
   '--captured-after 2026-10-01 --image-size original --max-api-pages 5 --workers 8')

In [ ]:
import json, collections
from pathlib import Path
manifest = json.loads(Path(f'/content/repo/data/{DATASET}/manifest.json').read_text())
images = [image for image in manifest['images'] if image.get('local_file')]
sizes = collections.Counter(f"{image.get('width')}x{image.get('height')}" for image in images)
leveled = sum(bool(image.get('camera_up')) for image in images)
print(f'{len(images)} panoramas downloaded; sizes {dict(sizes)}; {leveled} with the camera rotation the viewer levels with')
if len(images) < 100:
    raise RuntimeError('Fewer panoramas than expected: rerun the download cell (it resumes) and read its messages.')

## 3. Analysis

Segmentation, building instances, OSM footprints and association for every panorama, on the GPU. On a T4 it takes
roughly 10 to 25 seconds per panorama, under an hour in all. If the session drops, run the cells of section 1 again
(not the download), then this one: finished panoramas are skipped and the stage cache on Drive is reused.

In [ ]:
sh(f'uv run city-analyser-analyze {DATASET} --report {WORK}/analysis-report-{DATASET}.json')

In [ ]:
analyses = [path for path in Path(f'/content/repo/data/{DATASET}/analysis').glob('*.json') if not path.name.endswith('-sfm.json')]
buildings = collections.Counter(record['osm_id'] for path in analyses for record in json.loads(path.read_text())['building_ids'].values())
print(f'{len(analyses)} of {len(images)} panoramas analyzed; {len(buildings)} buildings matched, '
      f'{sum(count >= 2 for count in buildings.values())} of them seen from two panoramas or more')

## 4. Results to Drive

Two files on `MyDrive/city-analyser/`: the results with the panoramas, and the model-stage cache. They are large:
take them from Drive (or Google Drive for desktop) rather than through the browser download.

In [ ]:
sh(f'uv run city-analyser-results export {DATASET} {WORK}/results-{DATASET}.zip --with-images')
sh(f'cd {WORK} && rm -f stage-cache-{DATASET}.zip && zip -qr stage-cache-{DATASET}.zip stage-cache-{DATASET} && ls -lh results-{DATASET}.zip stage-cache-{DATASET}.zip', cwd='/content')

## 5. On the laptop

Put both zip files in the root of the repository and tell Claude. It imports them
(`city-analyser-results import results-madrid-salesas.zip`, and the stage cache into `data/.stage-cache/`), checks the
import, then does the mask review, the answers and the curation for the demo.